# v1.5.2 — BioBERT embedding of HINT criteria text (Colab GPU)

Run this notebook on Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU).

**Inputs you provide:**
1. Upload `training_dataset.parquet` (the Day 1 output, ~14 MB) via the file panel on the left.

**Outputs you download:**
1. `embeddings.npy` (~35 MB; the 11.5K × 768-dim mean-pooled BioBERT embeddings)
2. `embeddings_nctid.txt` (the row-index → NCT-ID alignment file)

**Wall-clock estimate:** ~5–10 minutes on T4 GPU; ~15 minutes on the smaller free-tier K80.

**Cost:** Free if you stay within Colab's free-tier GPU allowance; otherwise the model + data fit comfortably in one Colab Pro session.

## 1. Install dependencies

torch + transformers come pre-installed on most Colab runtimes; this pin avoids any version surprises. pyarrow is for the parquet read.

In [ ]:
!pip install -q 'transformers>=4.40' 'torch>=2.2' 'pyarrow>=15.0' 'pandas>=2.2' 'huggingface-hub>=0.20'
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Upload the parquet

If you don't see a file upload dialog, run `files.upload()` directly in another cell, or drag-and-drop `training_dataset.parquet` into the Colab file panel (left sidebar).

In [ ]:
import os
if not os.path.exists('training_dataset.parquet'):
    from google.colab import files
    print('Upload training_dataset.parquet (~14 MB)…')
    files.upload()

import pandas as pd
df = pd.read_parquet('training_dataset.parquet')
print(f'Loaded {len(df)} trials. Columns: {list(df.columns)}')
df[['nctid', 'phase', 'therapeutic_area', 'modality', 'label']].head()

## 3. Run BioBERT embeddings

Mean-pooled token embeddings (not CLS — see `embed_biobert.py` docstring for the design choice). Wordpiece-truncation from the back at 512 tokens.

Batch size 64 fits comfortably in T4's 16 GB VRAM.

In [ ]:
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_ID = 'dmis-lab/biobert-v1.1'
MAX_LENGTH = 512
BATCH_SIZE = 64
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading {MODEL_ID} on {device}…')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()
print('Model loaded.')

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

criteria_texts = df['criteria_text'].tolist()
nctids = df['nctid'].tolist()
n = len(criteria_texts)
embeddings = np.zeros((n, 768), dtype=np.float32)

with torch.no_grad():
    for start in range(0, n, BATCH_SIZE):
        end = min(start + BATCH_SIZE, n)
        batch_texts = criteria_texts[start:end]
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt',
        ).to(device)
        out = model(**enc)
        pooled = mean_pool(out.last_hidden_state, enc['attention_mask'])
        embeddings[start:end] = pooled.cpu().numpy().astype(np.float32)
        if (start // BATCH_SIZE) % 10 == 0:
            pct = end / n * 100
            print(f'  {end}/{n} embedded ({pct:.1f}%)')

print(f'\nDone. Embeddings shape: {embeddings.shape}')

## 4. Save and download

Two files written:
- `embeddings.npy` — the [N, 768] float32 array
- `embeddings_nctid.txt` — the row-index → NCT-ID map (alignment to the parquet)

In [ ]:
np.save('embeddings.npy', embeddings)
with open('embeddings_nctid.txt', 'w') as f:
    f.write('\n'.join(nctids) + '\n')

import os
print(f'embeddings.npy: {os.path.getsize("embeddings.npy") / 1024 / 1024:.2f} MB')
print(f'embeddings_nctid.txt: {os.path.getsize("embeddings_nctid.txt") / 1024:.2f} KB')

from google.colab import files
files.download('embeddings.npy')
files.download('embeddings_nctid.txt')
print('\nDownloaded both files. Place them at api/data/training/ on your local machine.')

## 5. Sanity check

Quick correctness check before you close the notebook: similar criteria text should produce embeddings with high cosine similarity; unrelated text should be near orthogonal.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Find two oncology trials and one CV trial; compare cosine similarities
onc_idx = df.index[df['therapeutic_area'] == 'oncology'][:2].tolist()
cv_idx = df.index[df['therapeutic_area'] == 'cardiovascular'][:1].tolist()

if len(onc_idx) >= 2 and len(cv_idx) >= 1:
    onc_emb = embeddings[onc_idx]
    cv_emb = embeddings[cv_idx]
    onc_self = cosine_similarity(onc_emb)[0, 1]
    onc_cv = cosine_similarity(onc_emb[0:1], cv_emb)[0, 0]
    print(f'Oncology trial 1 ↔ oncology trial 2 cosine sim: {onc_self:.3f}')
    print(f'Oncology trial 1 ↔ cardiovascular trial  cosine sim: {onc_cv:.3f}')
    if onc_self > onc_cv:
        print('✓ same-TA embeddings are closer than cross-TA — sanity check passes.')
    else:
        print('⚠ same-TA embeddings are NOT closer than cross-TA — something is off.')